In [ ]:
import fiftyone

/home/arun/.venvs/tshirt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset = fiftyone.zoo.load_zoo_dataset(
    "coco-2017",
    split="train",
    # label_types=["detections"],
    classes=["person"],
    max_samples=100,
)

 100% |██████|    1.9Gb/1.9Gb [1.7m elapsed, 0s remaining, 12.6Mb/s]        
Extracting annotations to '/home/arun/fiftyone/coco-2017/raw/instances_train2017.json'
 100% |██████████████████| 200/200 [1.3m elapsed, 0s remaining, 1.1 images/s]      
Writing annotations for 200 downloaded samples to '/home/arun/fiftyone/coco-2017/train/labels.json'
Dataset info written to '/home/arun/fiftyone/coco-2017/info.json'
Loading existing dataset 'coco-2017-train-200'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


In [ ]:
# Export the loaded COCO 'person' subset to a YOLO/Darknet-style directory
export_dir = './data/coco_person_yolo'
import os
os.makedirs(export_dir, exist_ok=True)
print('Exporting dataset to', export_dir)
# Try common FiftyOne YOLO export types; fall back with informative message
try:
    dataset.export(export_dir=export_dir, dataset_type=fiftyone.types.YOLOv4Dataset)
    print('Exported using fiftyone.types.YOLOv4Dataset')
except Exception:
    try:
        dataset.export(export_dir=export_dir, dataset_type=fiftyone.types.YOLOv5Dataset)
        print('Exported using fiftyone.types.YOLOv5Dataset')
    except Exception as e:
        print('Automatic export failed:', e)
        print('If export fails, please export the dataset to YOLO/Darknet format manually using FiftyOne or other tools.')

In [ ]:
# Prepare Darknet-style training files: train.txt, obj.names, obj.data
import glob
images_dir = os.path.join(export_dir, 'images')
labels_dir = os.path.join(export_dir, 'labels')
img_paths = sorted(glob.glob(os.path.join(images_dir, '**', '*.jpg'), recursive=True))
if not img_paths:
    img_paths = sorted(glob.glob(os.path.join(images_dir, '*.jpg')))
train_txt = os.path.join(export_dir, 'train.txt')
with open(train_txt, 'w') as f:
    for p in img_paths:
        f.write(p + '\n')
print('Wrote', train_txt, 'with', len(img_paths), 'images')
# obj.names: one class 'person'
names_path = os.path.join(export_dir, 'obj.names')
with open(names_path, 'w') as f:
    f.write('person\n')
# obj.data: classes, train path, names path, backup dir
data_path = os.path.join(export_dir, 'obj.data')
with open(data_path, 'w') as f:
    f.write('classes = 1\n')
    f.write('train = {}\n'.format(train_txt))
    f.write('names = {}\n'.format(names_path))
    f.write('backup = backup/')
print('Created obj.names and obj.data in', export_dir)

### Fine-tune YOLOv2 (Darknet) on the exported `person` dataset
Run the steps below on a Linux machine with the required build tools (gcc, make) and optional CUDA for GPU training. Replace `./data/coco_person_yolo` with the absolute path if required.

1. Clone and build Darknet (AlexeyAB)
2. Download pre-trained YOLOv2 weights
3. Create a `cfg/yolov2_person.cfg` from `cfg/yolov2.cfg`: set `classes=1` in the last [yolo] layer(s) and update the preceding convolutional layer's `filters` to `(classes + 5)*5 = 30`
4. Run `./darknet detector train` with the generated `obj.data` and the modified cfg, using the pre-trained `yolov2.weights` as a starting point.

Example commands (do not run inside this notebook unless you intend to build Darknet):
```bash
git clone https://github.com/AlexeyAB/darknet.git
cd darknet
# edit Makefile: set GPU=1, CUDNN=1, OPENCV=1 if available
make
wget https://pjreddie.com/media/files/yolov2.weights -O yolov2.weights
# copy cfg/yolov2.cfg -> cfg/yolov2_person.cfg and edit classes/filters as described above
./darknet detector train {export_dir}/obj.data cfg/yolov2_person.cfg yolov2.weights
```

Notes:
- If you prefer a PyTorch-based workflow, you can convert the exported YOLO labels and use a PyTorch YOLOv2 implementation to fine-tune similarly.
- Training will produce backup weights in the `backup/` folder defined in `obj.data`.